In [1]:

import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='Qwen/Qwen3-Omni-30B-A3B-Instruct',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

D:\Work\Langchain-learn-demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 定义提示词模板
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ('system', '请将下面的内容翻译为{language}'),
    ('user', "{text}")
])


# 使用基础解析器

In [3]:
chain = prompt_template | model
chain.invoke({'text': "今天天气不错", 'language': '法语'})

AIMessage(content="Il fait belle aujourd'hui.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 24, 'total_tokens': 30, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9e3086e62f489a23fe0887c7a755', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9e30-8d38-7312-a8cc-186832d56744-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 24, 'output_tokens': 6, 'total_tokens': 30, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt_template | model | StrOutputParser()
chain.invoke({'text': "今天天气不错", 'language': '法语'})

"Il fait belle aujourd'hui."

# 结构化

In [6]:
from pydantic import BaseModel, Field


# 使用模型
class Content(BaseModel):
    translated_text: str = Field(..., description="翻译后的文本")
    language: str = Field(..., description="翻译的语言")
    sentiment: str = Field(..., description="内容情绪")


chain = prompt_template | model.with_structured_output(Content)
chain.invoke({'text': "今天天气不错", 'language': '法语'})

Content(translated_text="Il fait belle aujourd'hui.", language='fr-FR', sentiment='Positif')

In [10]:
# 使用function call 本质上也是 with_structured_output

# deepseek
deepseek = ChatOpenAI(
    model='deepseek-ai/DeepSeek-V3.2',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)
class ResponseFormatter(BaseModel):
    """始终使用本工具来格式化输出"""
    translated_text: str = Field(..., description="翻译后的文本")
    language: str = Field(..., description="翻译的语言")
    sentiment: str = Field(..., description="内容情绪")
chain = prompt_template | deepseek.bind_tools([ResponseFormatter])
chain.invoke({'text': "今天天气不错", 'language': '法语'})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 83, 'prompt_tokens': 365, 'total_tokens': 448, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 365}, 'model_provider': 'openai', 'model_name': 'deepseek-ai/DeepSeek-V3.2', 'system_fingerprint': '', 'id': '019d9e43578c30d9f0ab3acee5a8936e', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d9e43-5e24-7d32-b586-28c52cd0631e-0', tool_calls=[{'name': 'ResponseFormatter', 'args': {'translated_text': "Le temps est beau aujourd'hui", 'language': '法语', 'sentiment': '积极'}, 'id': '019d9e4369940c4d205c227b96db9119', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 365, 'output_tokens': 83, 'total_tokens': 448, 'input_

In [7]:
from langchain_core.output_parsers import SimpleJsonOutputParser

# 使用提示词
# 创建聊天提示模板，要求模型以特定格式回答问题
prompt = ChatPromptTemplate.from_template(
    "尽你所能回答用户的问题。"  # 基本指令
    '你必须始终输出一个包含"answer"和"followup_question"键的JSON对象。其中"answer"代表：对用户问题的回答；"followup_question"代表：用户可能提出的后续问题'
    "{question}"  # 用户问题占位符
)


chain = prompt | model | SimpleJsonOutputParser()
resp = chain.invoke({"question": "细胞的动力源是什么？"})
resp

{'answer': '细胞的动力源是ATP（三磷酸腺苷）。ATP是细胞内直接提供能量的分子，通过水解分解成ADP（二磷酸腺苷）和无机磷酸，释放出能量，供细胞的各种生命活动使用，例如细胞分裂、物质运输、蛋白质合成等。',
 'followup_question': 'ATP是如何在细胞中生成的？'}